# Few-shot Inference with RAG — GQA & VQAv2

Runs RAG-augmented few-shot inference on either the GQA or VQAv2 Romanian
test sets. Switch dataset by setting `DATASET = 'gqa'` or `'vqav2'`.

**Pipeline:**
1. Load the ChromaDB RAG index built in notebook `03`
2. For each test question, retrieve `k` similar (question, image) demonstrations
   using a hybrid text+image score
3. Build a multimodal few-shot prompt
4. Run `FastVisionModel` (Unsloth, 4-bit) and append predictions to a JSONL
5. Evaluate with Romanian-aware normalization

The model stays loaded between dataset switches.

## 1. Install

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
!pip install -q chromadb sentence-transformers scikit-learn
import torch; torch._dynamo.config.recompile_limit = 64

## 2. Mount Drive + config

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

In [ ]:
# ===== Choose test set: 'gqa' or 'vqav2' =====
DATASET = 'gqa'

# RAG index built in notebook 03
RAG_DB_DIR = "/content/drive/My Drive/vqa_clean/rag_db_900k_v2"

# Base or fine-tuned model. Examples:
#   "unsloth/qwen3-vl-32b-instruct-unsloth-bnb-4bit"
#   "unsloth/gemma-4-31b-it-unsloth-bnb-4bit"
#   "<your-hf-username>/<adapter-name>"
MODEL_PATH = "unsloth/qwen3-vl-32b-instruct-unsloth-bnb-4bit"

# GQA images (also used as the few-shot demonstration pool)
GQA_IMAGES_ZIP = "/content/drive/My Drive/licenta/images.zip"

# Per-dataset paths
if DATASET == 'gqa':
    TEST_JSON       = "/content/drive/My Drive/vqa_clean/dataset_final/test_dataset_final_corrected_sure.json"
    TEST_IMAGES_ZIP = GQA_IMAGES_ZIP
    TEST_IS_LIST    = False
    SAVE_PATH       = "/content/drive/My Drive/licenta/predictions/unsloth_rag_GQA.jsonl"
else:
    TEST_JSON       = "/content/drive/My Drive/VQA_Val_Subset_15k/val_subset_15k_retranslated_gemma4_bigmodel.json"
    TEST_IMAGES_ZIP = "/content/drive/My Drive/VQA_Val_Subset_15k/val_subset_15k_archive.zip"
    TEST_IS_LIST    = True
    SAVE_PATH       = "/content/drive/My Drive/licenta/predictions/unsloth_rag_VQAv2.jsonl"

# Retrieval + generation
K_SHOTS      = 4
ALPHA        = 0.5        # weight for image score (0 = text only, 1 = image only)
FEWSHOT_MODE = 'multimodal'  # 'multimodal' = include demo images, 'text_only' = only demo Q/A
MAX_NEW_TOK  = 32

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

## 3. Unzip images

In [ ]:
import glob

def unzip_find(zip_path, dest, candidate_subdirs):
    for c in candidate_subdirs:
        if os.path.isdir(c) and glob.glob(os.path.join(c, "*.jpg")):
            print(f"Already in {c}")
            return c
    any_jpg = glob.glob(os.path.join(dest, "**/*.jpg"), recursive=True)
    if any_jpg:
        d = os.path.dirname(any_jpg[0])
        print(f"Already in {d}")
        return d
    print(f"Unzipping {zip_path}...")
    os.makedirs(dest, exist_ok=True)
    os.system(f'unzip -q -o "{zip_path}" -d "{dest}"')
    for c in candidate_subdirs:
        if os.path.isdir(c) and glob.glob(os.path.join(c, "*.jpg")):
            return c
    any_jpg = glob.glob(os.path.join(dest, "**/*.jpg"), recursive=True)
    return os.path.dirname(any_jpg[0]) if any_jpg else None

GQA_IMAGE_DIR  = unzip_find(GQA_IMAGES_ZIP, "/content/data/gqa",
                            ["/content/data/gqa/images", "/content/data/gqa"])
TEST_IMAGE_DIR = unzip_find(TEST_IMAGES_ZIP, f"/content/data/{DATASET}",
                            [f"/content/data/{DATASET}/images",
                             f"/content/data/{DATASET}/val2014",
                             f"/content/data/{DATASET}"])
print(f"GQA images:  {GQA_IMAGE_DIR}")
print(f"Test images: {TEST_IMAGE_DIR}")

## 4. Load RAG index + embedding models

In [ ]:
import json
import shutil
import chromadb
import torch
from sentence_transformers import SentenceTransformer

# Copy RAG to local disk for speed
LOCAL_RAG = "/content/rag_local"
if not os.path.exists(LOCAL_RAG):
    print("Copying RAG locally...")
    shutil.copytree(RAG_DB_DIR, LOCAL_RAG)

with open(os.path.join(LOCAL_RAG, "manifest.json"), encoding="utf-8") as f:
    manifest = json.load(f)

client = chromadb.PersistentClient(path=LOCAL_RAG)
collections = [c.name for c in client.list_collections()]
assert "questions" in collections and "images" in collections, "RAG index incomplete!"

q_collection   = client.get_collection("questions")
img_collection = client.get_collection("images")
print(f"RAG: {q_collection.count():,} questions | {img_collection.count():,} images")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
text_model  = SentenceTransformer(manifest["text_model"],  device=DEVICE)
image_model = SentenceTransformer(manifest["image_model"], device=DEVICE)

## 5. Hybrid retrieval (text + image)

In [ ]:
def retrieve_fewshot(question_text, query_image=None, k=4, alpha=0.5,
                     pool=50, exclude_image_id=None):
    scores, meta_by_id = {}, {}

    q_emb = text_model.encode([question_text], normalize_embeddings=True).tolist()
    rt = q_collection.query(query_embeddings=q_emb, n_results=pool)
    for meta, d in zip(rt["metadatas"][0], rt["distances"][0]):
        if exclude_image_id and meta.get("imageId") == exclude_image_id:
            continue
        qid = meta["questionId"]
        meta_by_id[qid] = meta
        w = (1 - alpha) if query_image is not None else 1.0
        scores[qid] = scores.get(qid, 0.0) + w * (1.0 - d / 2.0)

    if query_image is not None:
        i_emb = image_model.encode([query_image], normalize_embeddings=True).tolist()
        ri = img_collection.query(query_embeddings=i_emb, n_results=pool)
        # Image hits give scores to all questions sharing that image
        seen_imgs = {m["imageId"]: d for m, d in zip(ri["metadatas"][0], ri["distances"][0])}
        for qid, meta in meta_by_id.items():
            if meta.get("imageId") in seen_imgs:
                d = seen_imgs[meta["imageId"]]
                scores[qid] = scores.get(qid, 0.0) + alpha * (1.0 - d / 2.0)

    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return [meta_by_id[qid] for qid, _ in top]

## 6. Image helpers + answer cleaning

In [ ]:
from PIL import Image
import re

def load_gqa_img(image_id):
    p = os.path.join(GQA_IMAGE_DIR, str(image_id) + ".jpg")
    if os.path.exists(p):
        try: return Image.open(p).convert("RGB")
        except Exception: return None
    return None

def load_test_img(image_id, image_filename=None):
    if DATASET == "gqa":
        p = os.path.join(TEST_IMAGE_DIR, str(image_id) + ".jpg")
        if os.path.exists(p):
            try: return Image.open(p).convert("RGB")
            except Exception: return None
        return None
    if image_filename:
        p = os.path.join(TEST_IMAGE_DIR, image_filename)
        if os.path.exists(p):
            try: return Image.open(p).convert("RGB")
            except Exception: pass
    p = os.path.join(TEST_IMAGE_DIR, f"COCO_val2014_{int(image_id):012d}.jpg")
    if os.path.exists(p):
        try: return Image.open(p).convert("RGB")
        except Exception: return None
    return None

def clean_answer(text):
    text = text.strip().split("\n")[0].strip()
    text = re.sub(r"(?i)^.*?r[ăa]spuns\s*:?\s*", "", text).strip()
    text = text.split(" ")[0].strip()
    text = re.sub(r"[^\wăâîșțĂÂÎȘȚ]", "", text)
    return text.lower()

## 7. Load model

In [ ]:
!pip install --upgrade "torchao>=0.16.0"

from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    model_name   = MODEL_PATH,
    load_in_4bit = True,
)
FastVisionModel.for_inference(model)
print(f"Model loaded: {MODEL_PATH}")

## 8. Few-shot prompt + inference

In [ ]:
INSTRUCTION = "Răspunde la întrebare cu un singur cuvânt în română."

def build_messages(question, test_image, demos, mode):
    content = [{"type": "text", "text": INSTRUCTION}]
    images_for_tokenizer = []
    for d in demos:
        if mode == "multimodal":
            dimg = load_gqa_img(d["imageId"])
            if dimg is not None:
                content.append({"type": "image"})
                images_for_tokenizer.append(dimg)
        content.append({"type": "text",
                        "text": f"Întrebare: {d['question']} Răspuns: {d['answer']}"})
    content.append({"type": "image"})
    images_for_tokenizer.append(test_image)
    content.append({"type": "text", "text": f"Întrebare: {question} Răspuns:"})
    return [{"role": "user", "content": content}], images_for_tokenizer

def run_fewshot(question, test_image, demos, mode="multimodal"):
    messages, images = build_messages(question, test_image, demos, mode)
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(images, input_text, return_tensors="pt",
                       add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOK,
                             do_sample=False, use_cache=True)
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    return decoded.split("Răspuns:")[-1].strip()

## 9. Load test set

In [ ]:
with open(TEST_JSON, encoding="utf-8") as f:
    test_raw = json.load(f)

test_items = []
if TEST_IS_LIST:
    for v in test_raw:
        test_items.append({
            "questionId":     str(v["question_id"]),
            "imageId":        str(v["imageId"]),
            "image_filename": v.get("image_filename", ""),
            "question":       v.get("question_retranslated") or v.get("question", ""),
            "gt_answer":      v.get("answer_retranslated")   or v.get("answer", ""),
        })
else:
    for qid, v in test_raw.items():
        test_items.append({
            "questionId":     str(qid),
            "imageId":        str(v.get("imageId", "")),
            "image_filename": "",
            "question":       v.get("question_retranslated") or v.get("question", ""),
            "gt_answer":      v.get("answer_retranslated")   or v.get("answer_clean") or v.get("answer", ""),
        })
print(f"Test examples: {len(test_items)}")

## 10. Quick sanity check on 5 examples

In [ ]:
print("=" * 70)
for i, ex in enumerate(test_items[:5]):
    img = load_test_img(ex["imageId"], ex.get("image_filename"))
    if img is None:
        print(f"[{i+1}] image missing"); continue
    demos = retrieve_fewshot(ex["question"], query_image=img, k=K_SHOTS,
                             alpha=ALPHA, exclude_image_id=ex["imageId"])
    pred  = run_fewshot(ex["question"], img, demos, mode=FEWSHOT_MODE)
    match = "✓" if clean_answer(pred) == clean_answer(ex["gt_answer"]) else "✗"
    print(f"[{i+1}] {match} Q: {ex['question']}")
    print(f"     GT: {ex['gt_answer']}  |  Pred: {pred}")
print("=" * 70)

## 11. Full inference (resume-safe)

In [ ]:
from tqdm.auto import tqdm

processed = set()
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, encoding="utf-8") as f:
        for line in f:
            try: processed.add(json.loads(line)["questionId"])
            except Exception: pass
print(f"Already processed: {len(processed):,} / {len(test_items):,}")

skipped = 0
with open(SAVE_PATH, "a", encoding="utf-8") as out:
    for ex in tqdm(test_items, desc=f"RAG few-shot {DATASET}"):
        qid = ex["questionId"]
        if qid in processed:
            continue
        try:
            image = load_test_img(ex["imageId"], ex.get("image_filename"))
            if image is None:
                skipped += 1
                continue
            demos = retrieve_fewshot(ex["question"], query_image=image,
                                     k=K_SHOTS, alpha=ALPHA,
                                     exclude_image_id=ex["imageId"])
            pred  = run_fewshot(ex["question"], image, demos, mode=FEWSHOT_MODE)
            out.write(json.dumps({
                "questionId": qid,
                "imageId":    ex["imageId"],
                "question":   ex["question"],
                "gt_answer":  ex["gt_answer"],
                "prediction": pred,
                "demos":      [{"q": d["question"], "a": d["answer"]} for d in demos],
            }, ensure_ascii=False) + "\n")
            out.flush()
            processed.add(qid)
        except Exception as e:
            print(f"Error on {qid}: {e}")
            continue

print(f"Done. Skipped (missing image): {skipped}")

## 12. Evaluate (Romanian-aware normalization)

In [ ]:
import unicodedata
from sklearn.metrics import accuracy_score, f1_score

def clean_text(t):
    if not t:
        return ""
    t = unicodedata.normalize("NFC", str(t).lower().strip())
    return re.sub(r"\s+", " ", t).strip().strip(".")

# Romanian gender/number canonicalization for colors and a few common adjectives
CANON = {
    "albă":"alb","albe":"alb","albi":"alb",
    "neagră":"negru","negre":"negru","negri":"negru",
    "roșie":"roșu","roșii":"roșu","rosu":"roșu","rosie":"roșu",
    "albastră":"albastru","albastre":"albastru","albaștri":"albastru",
    "verzi":"verde",
    "galbenă":"galben","galbene":"galben","galbeni":"galben",
    "portocalie":"portocaliu","portocalii":"portocaliu",
}
def normalize(t):
    t = clean_text(t)
    return " ".join(CANON.get(w, w) for w in t.split())

preds = []
with open(SAVE_PATH, encoding="utf-8") as f:
    for line in f:
        try: preds.append(json.loads(line))
        except Exception: continue

gt   = [normalize(p["gt_answer"])  for p in preds]
pred = [normalize(p["prediction"]) for p in preds]

print(f"Examples: {len(preds)}")
print(f"Accuracy (normalized): {accuracy_score(gt, pred):.4f}")
print(f"F1 weighted:           {f1_score(gt, pred, average='weighted'):.4f}")

## How to run both datasets

1. Set `DATASET = 'gqa'`, run cells 2-12 → produces `unsloth_rag_GQA.jsonl`
2. Set `DATASET = 'vqav2'` (cell 2), then re-run config + cells 3, 9, 11, 12
   (model stays loaded)

**Notes:**
- The RAG index is copied locally on first run for stability (no Drive latency)
- If `list_collections() == []`, rebuild the index with notebook `03`
- `FEWSHOT_MODE='text_only'` skips demo images — much faster, slightly weaker